In [1]:
import pandas as pd
import os
import csv
import re
import ftfy
import unicodedata
from sympy import Union
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE, Unigram
from tokenizers.pre_tokenizers import ByteLevel, Whitespace
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer, UnigramTrainer
from tokenizers.processors import TemplateProcessing
from sklearn.model_selection import train_test_split


In [2]:
# data = pd.read_csv(r'..\data\processed\final_combined_jokes.csv')
data = pd.read_csv(r'../data/combined_data/final_combined_jokes.csv')
data.head()

,rid,stable_id,joke,topic
0,1,0000066daaacde54c609a69be4f00c3336480137,Whoever said imitation is the sincerest form o...,"imitation, flattery, minute"
1,2,00000a79878b3729adc0e47a34dbf5d5484214c0,"You're so fat, they oughta call your dick gary...","oldman, dick, gary"
2,3,000054bc76448f9b302e6266a72324ecb81d4083,I couldn't sleep last night. I was wondering w...,"night, sun"
3,4,0000b76ab05f5d5cf9952d109c5ce7b143ae016f,What's 11 & 2? The Cowboys,cowboys
4,5,00015c7571451cc5eb99c24dd7bc46a5ce46b9a1,I just got done apologizing to my barbershop q...,"barbershop, quartet, bucket"


In [3]:
def clean_jokes(joke):
    """Remove noise and normalize text for tokenization."""
    # Fix encoding
    joke = ftfy.fix_text(str(joke))
    joke = unicodedata.normalize("NFKC", joke)
    
    # Lowercase
    joke = joke.lower()
    
    # Remove URLs, subreddit refs, mentions, hashtags
    joke = re.sub(r'https?://\S+|www\.\S+', '', joke)
    joke = re.sub(r'/r/\S+', '', joke)
    joke = re.sub(r'@\w+|#\w+', '', joke)
    
    # Keep letters, numbers, and common punctuation
    joke = re.sub(r"[^a-z0-9\s.,!?;:'\"()\-\[\]""''…]", " ", joke)
    
    joke = re.sub(r'\.(?:\s*\.)+', '<ELLIPSIS>', joke)
    
    # Reduce repeated punctuation
    joke = re.sub(r'([!?])\1+', r'\1', joke)
         
    # Normalize whitespace
    joke = ' '.join(joke.split())
    
    # Protect contractions
    joke = re.sub(r"(\w)'(\w)", r"\1<APOST>\2", joke)
    
    # Add spacing around punctuation
    joke = re.sub(r"([.!?,;:(){}\[\]\"""''])", r" \1 ", joke)
    
    # Collapse extra spaces
    joke = re.sub(r"\s{2,}", " ", joke).strip()
    
    # Restore apostrophes
    joke = joke.replace("<APOST>", "'")
    
    joke = joke.replace("<ELLIPSIS>", " ... ")
    
    # Remove space around apostrophes (edge cases)
    joke = re.sub(r"\s+'\s+", "'", joke)
    
    return joke.strip()
data["joke_cleaned"] = data.joke.apply(lambda x: clean_jokes(x))
data = data[data.joke_cleaned.notnull()]

In [4]:
print(clean_jokes("What no rEally?!! hahaha... . ..!!!!"))

what no really ? ! hahaha ...  !


In [5]:
data.joke_cleaned.to_list()[:10]

["whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .",
 "you're so fat , they oughta call your dick gary oldman  ... cause it always disappears into a roll .",
 "i couldn't sleep last night . i was wondering where the sun went then it dawned on me",
 "what's 11 2 ? the cowboys",
 'i just got done apologizing to my barbershop quartet i gathered them to sing a song about a bucket with a lot of water in it . it turned out to be solo .',
 'how do you know that a russian oligarch is serious about a deal ? he went to jared .',
 'don\'t expect a " bless you " after the 4th sneeze ... get your shit together .',
 'dj scratches a sick mix crowd goes wild dj scratches a puppy\'s ear crowd " awws " dj scratches lotto ticket crowd " oohs " wins 1',
 'my girlfriend told me she enjoys celebrity impressions in bed , tonight i tried jim carrey apparentley " like a glove " is crossing the line',
 "i was in a cab and the cab driv

In [6]:
data["word_count"] = data.joke_cleaned.apply(lambda x: len(x.split()))

In [7]:
full_text = " ".join(data.joke_cleaned.to_list())

In [8]:
total_words = full_text.split()
unique_words = set(total_words)
print(f"""
total characters  : {len(full_text)}
total words       : {len(total_words)}
total unique words: {len(unique_words)}
""")


total characters  : 108058070
total words       : 23067226
total unique words: 185856



In [9]:
freq = Counter(total_words)

In [10]:
freq.most_common(10)

[('.', 1006802),
 ('the', 951721),
 ('a', 788794),
 (',', 728398),
 ('"', 718161),
 ('to', 449766),
 ('i', 440778),
 ('and', 420686),
 ('?', 405957),
 ('you', 337722)]

In [11]:
# data.to_csv(r'..\data\processed\final_cleaned_jokes.csv', index=False)
data.to_csv(r'../data/processed/final_cleaned_jokes.csv', index=False)

In [12]:
with open('../data/processed/jokes_text.txt','w') as f:
    f.write(full_text)

## BPE Tokenizer

In [13]:
with open('../data/processed/jokes_text.txt','r') as f:
    full_text = f.read()

In [14]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel()
# tokenizer.pre_tokenizer = Whitespace()

special_tokens = [
    "[S]",      # Start of sequence
    "[/S]",     # End of sequence
    "[PAD]",    # Padding
    "[UNK]",    # Unknown
    "[MASK]",   
    "[USER]",
    "[JOKE]"
]

trainer = BpeTrainer(
    vocab_size=30000,
    min_frequency=2,
    special_tokens=special_tokens
)
tokenizer.train(files=['../data/processed/jokes_text.txt'], trainer=trainer)


In [15]:
tokenizer.decoder = ByteLevelDecoder()

tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]")

tokenizer.enable_truncation(max_length=256,stride=0,strategy='longest_first',direction='right')

tokenizer.post_processor = TemplateProcessing(
    single="[S] $A [/S]",
    pair="[S] $A [JOKE] $B [/S]",
    special_tokens=[
        ("[S]", tokenizer.token_to_id("[S]")), 
        ("[JOKE]", tokenizer.token_to_id("[JOKE]")),
        ("[/S]", tokenizer.token_to_id("[/S]")),
    ],
)

In [16]:
output = tokenizer.encode("tell me a joke about elephants, rain, trunk", "this is a joke about elephants, rain, trunk")
output.tokens

['[S]',
 'Ġtell',
 'Ġme',
 'Ġa',
 'Ġjoke',
 'Ġabout',
 'Ġelephants',
 ',',
 'Ġrain',
 ',',
 'Ġtrunk',
 '[JOKE]',
 'Ġthis',
 'Ġis',
 'Ġa',
 'Ġjoke',
 'Ġabout',
 'Ġelephants',
 ',',
 'Ġrain',
 ',',
 'Ġtrunk',
 '[/S]']

In [17]:
tokenizer.get_vocab_size()

30000

In [18]:
tokenizer.save(r'../data/processed/tokenizer.json')

# Unigram SentencePiece Tokenizer 

In [19]:

# Avoid parallelism warning in notebooks
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SPECIAL_TOKENS = [
    "[S]",      # Start of sequence
    "[/S]",     # End of sequence
    "[PAD]",    # Padding
    "[UNK]",    # Unknown
    "[MASK]",   
    "[USER]",
    "[JOKE]"
]


def build_unigram_tokenizer(
    corpus_files,
    vocab_size=30_000,
    min_frequency=2,
    max_length=256,
 

):
    
    bos_token = "[S]"
    eos_token = "[/S]"
    unk_token = "[UNK]"
    pad_token = "[PAD]"

    tok = Tokenizer(Unigram())  # empty model; will be trained
    # tok.pre_tokenizer = Whitespace()
    tok.pre_tokenizer = ByteLevel(add_prefix_space=True)

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        unk_token=unk_token,
        special_tokens=SPECIAL_TOKENS,
        # Unigram is frequency-based; this helps stability on small corpora
        max_piece_length=16,
        shrink_factor=0.75,
        n_sub_iterations=2,
        # min_frequency prevents ultra-rare pieces from polluting vocab
        min_frequency=min_frequency,
    )

    tok.train(files=['../data/processed/jokes_text.txt'], trainer=trainer)
    tok.decoder = ByteLevelDecoder()

    
    # Post-processing to inject BOS/EOS like in the BPE pipeline
    tok.post_processor = TemplateProcessing(
        single=f"[S] $A [/S]",
        pair=f"[S] $A [JOKE] $B [/S]",
        special_tokens=[
            ("[S]", tok.token_to_id("[S]")),
            ("[/S]", tok.token_to_id("[/S]")),
            ("[JOKE]", tok.token_to_id("[JOKE]")),
        ],
    )

    tok.enable_truncation(max_length=max_length)
    tok.enable_padding(
        pad_id=tok.token_to_id("[PAD]"),
        pad_token="[PAD]",
        # pad_to_multiple_of=8,  # uncomment if you want tensor-core-friendly shapes
        direction="right",
        length=None,  # dynamic to the longest in batch
    )

    return tok

def save_tokenizer(tokenizer, out_path):
    from pathlib import Path
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer.save(str(out_path))

def load_tokenizer(path):
    from pathlib import Path
    return Tokenizer.from_file(str(Path(path)))

In [20]:
# Example usage
DATA_DIR = Path("../data/processed")        
CORPUS = [DATA_DIR / "jokes_text.txt"] 

unigram_tok = build_unigram_tokenizer(
    CORPUS,
    vocab_size=30_000,
    
)

save_tokenizer(unigram_tok, "../data/processed/tokenizer_unigram.json")
reloaded = load_tokenizer("../data/processed/tokenizer_unigram.json")

# Encode single + pair (mirrors your BPE examples)
enc_single = reloaded.encode("tell me a joke about elephants, rain, trunk")
enc_pair = reloaded.encode(
    "tell me a joke about elephants, rain, trunk",
    "this is a joke about elephants, rain, trunk"
)

print(enc_single.tokens[:20])
print(enc_pair.tokens[:25])


Ignored unknown kwargs option shrink_factor
Ignored unknown kwargs option min_frequency


['[S]', 'Ġtell', 'Ġme', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġelephant', 's', ',', 'Ġrain', ',', 'Ġtrunk', '[/S]']
['[S]', 'Ġtell', 'Ġme', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġelephant', 's', ',', 'Ġrain', ',', 'Ġtrunk', '[JOKE]', 'Ġthis', 'Ġis', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġelephant', 's', ',', 'Ġrain', ',']


In [21]:
from pathlib import Path
from tokenizers import Tokenizer

# === config ===
DATA_DIR = Path("../data/processed")
CORPUS = [DATA_DIR / "jokes_text.txt"]  # use your real corpus file
VOCAB_SIZE = 30_000
MAX_LEN = 256
ARTIFACT = DATA_DIR / "tokenizer_unigram.json"

# pick the right builder name for your notebook:
try:
    builder_fn = build_unigram_tokenizer
except NameError:
    builder_fn = build_unigram_tokenizer  # fallback if you kept the older name

# === tiny assert helpers (no pytest) ===
def check(name, cond, msg_ok="ok", msg_fail="fail"):
    if cond:
        print(f"✅ {name}: {msg_ok}")
    else:
        print(f"❌ {name}: {msg_fail}")

def ids_len(enc):
    return len(enc.ids)

# === build ===
tok = builder_fn(CORPUS, vocab_size=VOCAB_SIZE, max_length=MAX_LEN)

# 1) specials exist
specials = ["[S]","[/S]","[PAD]","[UNK]","[MASK]","[USER]","[JOKE]"]
missing = [t for t in specials if tok.token_to_id(t) is None]
check("special tokens in vocab", len(missing) == 0, msg_ok="all present", msg_fail=f"missing: {missing}")

# 2) single encode has [S] ... [/S]
s = "tell me a joke about elephants, rain, trunk"
e_single = tok.encode(s)
toks_single = e_single.tokens
check("single has BOS/EOS", toks_single[0] == "[S]" and toks_single[-1] == "[/S]")

# 3) pair encode auto-inserts [JOKE] between A/B (shared BOS/EOS template)
a = "tell me a joke about elephants, rain, trunk"
b = "this is a joke about elephants, rain, trunk"
e_pair = tok.encode(a, b)
toks_pair = e_pair.tokens

has_shared_bos = toks_pair[0] == "[S]"
has_shared_eos = toks_pair[-1] == "[/S]"
has_joke_between = "[JOKE]" in toks_pair
check("pair has shared BOS/EOS", has_shared_bos and has_shared_eos)
check("[JOKE] inserted between A and B", has_joke_between)

# 4) round-trip decode (ByteLevel decoder should give clean text)
dec_single = tok.decode(e_single.ids)
# We won't require exact equality because BOS/EOS aren't part of decoded text.
contains_all = all(w in dec_single for w in ["tell","joke","elephants","rain","trunk"])
check("round-trip contains key words", contains_all)

# 5) vocab size sanity (unigram can be slightly under target)
vsz = tok.get_vocab_size()
check("vocab size close", int(0.85 * VOCAB_SIZE) <= vsz <= VOCAB_SIZE, msg_ok=f"{vsz}", msg_fail=f"{vsz}")

# 6) truncation/padding behavior
# batch encode a few strings with very different lengths
batch = [
    "short one",
    "a bit longer sentence with some tokens in it",
    "extremely " + "long " * 400 + "string",  # should get truncated
]
enc_batch = tok.encode_batch(batch)

# verify max length <= MAX_LEN and all padded to same length
lengths = [ids_len(e) for e in enc_batch]
same_len = len(set(lengths)) == 1
within_max = max(lengths) <= MAX_LEN
pad_id = tok.token_to_id("[PAD]")
uses_pad = any(pad_id in e.ids for e in enc_batch)  # at least some will be padded

check("batch padded to same length", same_len, msg_ok=f"len={lengths[0]} for all")
check("batch length within MAX_LEN", within_max, msg_ok=f"max={max(lengths)}", msg_fail=f"max={max(lengths)} > {MAX_LEN}")
check("PAD id used in batch", uses_pad)

# 7) save / reload equivalence on decoding
ARTIFACT.parent.mkdir(parents=True, exist_ok=True)
tok.save(str(ARTIFACT))
reloaded = Tokenizer.from_file(str(ARTIFACT))
check(
    "save/reload decode equal",
    tok.decode(tok.encode(s).ids) == reloaded.decode(reloaded.encode(s).ids)
)

# 8) quick peek at tokens to visually confirm ByteLevel-style pieces
print("\nSingle tokens preview:", toks_single[:24])
print("Pair tokens preview:", toks_pair[:36])


Ignored unknown kwargs option shrink_factor
Ignored unknown kwargs option min_frequency


✅ special tokens in vocab: all present
✅ single has BOS/EOS: ok
✅ pair has shared BOS/EOS: ok
✅ [JOKE] inserted between A and B: ok
✅ round-trip contains key words: ok
✅ vocab size close: 30000
✅ batch padded to same length: len=256 for all
✅ batch length within MAX_LEN: max=256
✅ PAD id used in batch: ok
✅ save/reload decode equal: ok

Single tokens preview: ['[S]', 'Ġtell', 'Ġme', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġelephant', 's', ',', 'Ġrain', ',', 'Ġtrunk', '[/S]']
Pair tokens preview: ['[S]', 'Ġtell', 'Ġme', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġelephant', 's', ',', 'Ġrain', ',', 'Ġtrunk', '[JOKE]', 'Ġthis', 'Ġis', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġelephant', 's', ',', 'Ġrain', ',', 'Ġtrunk', '[/S]']
